# Module 02 — Lab: Messages API in depth

Covers: streaming, multi-turn, stop sequences, error handling, token counting.

In [ ]:
import os, time, json
from dotenv import load_dotenv
from anthropic import Anthropic, APIStatusError

load_dotenv('../.env')
client = Anthropic()
MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')

## 1. Streaming with TTFB measurement

In [ ]:
t0 = time.perf_counter()
ttfb = None
with client.messages.stream(
    model=MODEL, max_tokens=400,
    messages=[{'role':'user','content':'Tell me a 200-word story about a cartographer.'}]
) as stream:
    for text in stream.text_stream:
        if ttfb is None:
            ttfb = time.perf_counter() - t0
        print(text, end='', flush=True)
    final = stream.get_final_message()

total = time.perf_counter() - t0
print(f"\n\nTTFB: {ttfb*1000:.0f} ms   total: {total*1000:.0f} ms   out tokens: {final.usage.output_tokens}")

## 2. Multi-turn Conversation class

In [ ]:
class Conversation:
    def __init__(self, system, model=MODEL, max_tokens=1024):
        self.system = system
        self.model = model
        self.max_tokens = max_tokens
        self.messages = []

    def ask(self, user_text):
        self.messages.append({'role':'user','content':user_text})
        r = client.messages.create(
            model=self.model, max_tokens=self.max_tokens,
            system=self.system, messages=self.messages,
        )
        self.messages.append({'role':'assistant','content': r.content})
        return ''.join(b.text for b in r.content if b.type == 'text')

    def reset(self):
        self.messages = []

convo = Conversation('You are a terse senior engineer. Reply in <=2 sentences.')
print(convo.ask('What is the GIL?'))
print(convo.ask('And how does asyncio sidestep it?'))
print(f'\nhistory turns: {len(convo.messages)}')

## 3. Stop sequences with structured tags

In [ ]:
r = client.messages.create(
    model=MODEL, max_tokens=256,
    stop_sequences=['</answer>'],
    system='Wrap your final answer in <answer>...</answer>. Be concise.',
    messages=[{'role':'user','content':'What is the largest prime under 100?'}],
)
raw = r.content[0].text
print('stop_reason:', r.stop_reason)
print('raw:', raw)
# Extract content between <answer> and the (stripped) closing tag
import re
m = re.search(r'<answer>(.*)', raw, re.DOTALL)
print('answer:', m.group(1).strip() if m else None)

## 4. Retry wrapper with tenacity

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, before_sleep_log
import logging
logging.basicConfig(level=logging.INFO)

RETRYABLE = (APIStatusError,)

@retry(
    retry=retry_if_exception_type(RETRYABLE),
    wait=wait_exponential(min=1, max=20),
    stop=stop_after_attempt(5),
    before_sleep=before_sleep_log(logging.getLogger(), logging.WARNING),
    reraise=True,
)
def safe_create(**kw):
    return client.messages.create(**kw)

r = safe_create(model=MODEL, max_tokens=32, messages=[{'role':'user','content':'ping'}])
print(r.content[0].text)

## 5. Pre-flight token counting

In [ ]:
DOC = 'Lorem ipsum dolor sit amet, ' * 500  # ~5k tokens of filler

tc = client.messages.count_tokens(
    model=MODEL,
    system='Summarize precisely.',
    messages=[{'role':'user','content': DOC}],
)
print('input_tokens (estimated):', tc.input_tokens)